In [13]:
from huggingface_hub import login, notebook_login
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [15]:
HF_TOKEN = 'hf_REDACTED_ROTATE_THIS_TOKEN'
from huggingface_hub import login
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
# !huggingface-cli login --token hf_REDACTED_ROTATE_THIS_TOKEN --add-to-git-credential

In [17]:
PAD_TOKEN = "<|pad|>"
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
FINETUNED_MODEL_NAME = "harshil30402/Cognius-mini-1B-v1"

In [18]:
device = 'mps'

In [29]:
# GROQ_API_KEY = 'gsk_REDACTED_ROTATE_THIS_TOKEN'

GROQ_API_KEY = 'gsk_REDACTED_ROTATE_THIS_TOKEN'

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map=device,
)

In [12]:
# model_dir = "/content/drive/MyDrive/Cognius-1B"

# from transformers import AutoModelForCausalLM, AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained(model_dir)
# model = AutoModelForCausalLM.from_pretrained(model_dir)

In [ ]:
finetuned_tokenizer = AutoTokenizer.from_pretrained('harshil30402/Cognius-1B')
finetuned_tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
finetuned_tokenizer.padding_side = "right"

finetuned_model = AutoModelForCausalLM.from_pretrained(
    'harshil30402/Cognius-8B-v1',
    device_map=device,
)

Fetching 4 files:   0%|          | 0/4 [00:23<?, ?it/s]


In [32]:
# Pipelines

original_generation_pipeline = pipeline(
    task='text-generation',
    model=model,
    tokenizer=tokenizer,
)

finetuned_generation_pipeline = pipeline(
    task='text-generation',
    model=finetuned_model,
    tokenizer=finetuned_tokenizer,
)

Device set to use cuda
Device set to use cuda


In [31]:
prompt = """

You are an expert evaluator tasked with comparing responses from two AI systems (a standard model and a fine-tuned model) to the same question, and judging how close each is to the correct answer.

Question: {question}

Correct Answer:
{correct_answer}

Standard Model Response:
{standard_response}

Fine-tuned Model Response:
{finetuned_response}

Please evaluate both model responses on a scale of 1-10, where 1 means extremely far from the correct answer and 10 means perfectly aligned with the correct answer. Consider the following criteria:
- Accuracy and factual correctness compared to the ground truth
- Relevance to the question asked
- Completeness relative to the correct answer
- Clarity and coherence
- Reasoning quality
- Nuance and consideration of different perspectives
- Helpfulness to the user

First, analyze both model responses separately, comparing each to the correct answer. Identify how closely each aligns with the ground truth according to the criteria above.

Then, explain specifically what each model did well or poorly compared to the correct answer. Note which model came closer to providing the correct information and why.

Finally, provide a numerical score (1-10) for each model:
Standard Model Score: [1-10]
Fine-tuned Model Score: [1-10]

"""

In [25]:
import pandas as pd
test_df = pd.read_json('test.json')

In [26]:
test_df

,task,reasoning,self_reflection,answer
0,What is the longest river in the world?,The Nile River is often cited as the longest r...,I will include that some experts argue about t...,"Traditionally, the Nile River is considered th..."
1,How do vaccines work?,Vaccines work by stimulating the immune system...,I need to ensure that the mechanism of buildin...,Vaccines introduce a harmless component of a p...
2,What is gravity?,Gravity is the force that pulls objects toward...,Correct. I could also mention it was formulate...,Gravity is the force that attracts objects tow...
3,What is the significance of the Sistine Chapel...,"The Sistine Chapel ceiling, painted by Michela...",Referencing both the artist and the subject ma...,The Sistine Chapel ceiling is significant for ...
4,What is the difference between weather and cli...,Weather refers to the short-term atmospheric c...,"The distinction is correct, but I should empha...",Weather is the short-term atmospheric conditio...
...,...,...,...,...
100,What is the boiling point of water in Celsius?,The boiling point of water is 100 degrees Cels...,100 degrees Celsius is correct for the boiling...,The boiling point of water is 100 degrees Cels...
101,What is a virus?,A virus is a non-living infectious agent that ...,Clarify it’s acellular and differs from bacteria.,A virus is an acellular infectious agent that ...
102,What is the function of chlorophyll in plants?,Chlorophyll is the pigment responsible for abs...,I need to emphasize its crucial role in captur...,"Chlorophyll absorbs sunlight, which is then us..."
103,What is the difference between a concave and c...,A concave lens is thinner in the middle and di...,I should clarify that concave lenses are used ...,"A concave lens, used in glasses for nearsighte..."


In [41]:
model_responses = []
finetuned_responses = []

In [43]:
from tqdm import tqdm

In [43]:
def compare(question: str):
    original = original_generation_pipeline(question, max_new_tokens=100)[0]['generated_text']
    finetuned = finetuned_generation_pipeline(question, max_new_tokens=100)[0]['generated_text']
    return original, finetuned

for question in tqdm(test_df['task'], desc="Generating model responses", unit="question"):
    original, finetuned = compare(question)
    model_responses.append(original)
    finetuned_responses.append(finetuned)

Generating model responses: 100%|██████████| 105/105 [11:14<00:00,  6.42s/question]


In [45]:
# Add the responses to the DataFrame
test_df['model'] = model_responses
test_df['finetuned_model'] = finetuned_responses

In [ ]:
test_df.to_csv('data/testing_model_performances.csv', index=False)

In [49]:
import pandas as pd

comparison_df = pd.read_csv('data/testing_model_performances.csv')
comparison_df.head()

,task,reasoning,self_reflection,answer,model,finetuned_model
0,What is the longest river in the world?,The Nile River is often cited as the longest r...,I will include that some experts argue about t...,"Traditionally, the Nile River is considered th...",What is the longest river in the world? The Ni...,What is the longest river in the world? The Ni...
1,How do vaccines work?,Vaccines work by stimulating the immune system...,I need to ensure that the mechanism of buildin...,Vaccines introduce a harmless component of a p...,How do vaccines work? Vaccines contain weakene...,How do vaccines work? Vaccines are a safe and ...
2,What is gravity?,Gravity is the force that pulls objects toward...,Correct. I could also mention it was formulate...,Gravity is the force that attracts objects tow...,What is gravity? Gravity is a fundamental forc...,What is gravity? Gravity is a fundamental forc...
3,What is the significance of the Sistine Chapel...,"The Sistine Chapel ceiling, painted by Michela...",Referencing both the artist and the subject ma...,The Sistine Chapel ceiling is significant for ...,What is the significance of the Sistine Chapel...,What is the significance of the Sistine Chapel...
4,What is the difference between weather and cli...,Weather refers to the short-term atmospheric c...,"The distinction is correct, but I should empha...",Weather is the short-term atmospheric conditio...,What is the difference between weather and cli...,What is the difference between weather and cli...


In [50]:
# Pre-processing

comparison_df['model'] = comparison_df.apply(lambda row: row['model'].replace(row['task'], ''), axis=1)
comparison_df['finetuned_model'] = comparison_df.apply(lambda row: row['finetuned_model'].replace(row['task'], ''), axis=1)

comparison_df.head()

,task,reasoning,self_reflection,answer,model,finetuned_model
0,What is the longest river in the world?,The Nile River is often cited as the longest r...,I will include that some experts argue about t...,"Traditionally, the Nile River is considered th...",The Nile River is often considered the longes...,The Nile River is the longest river in the wo...
1,How do vaccines work?,Vaccines work by stimulating the immune system...,I need to ensure that the mechanism of buildin...,Vaccines introduce a harmless component of a p...,Vaccines contain weakened or inactivated path...,Vaccines are a safe and effective way to prev...
2,What is gravity?,Gravity is the force that pulls objects toward...,Correct. I could also mention it was formulate...,Gravity is the force that attracts objects tow...,Gravity is a fundamental force of nature that...,Gravity is a fundamental force of nature that...
3,What is the significance of the Sistine Chapel...,"The Sistine Chapel ceiling, painted by Michela...",Referencing both the artist and the subject ma...,The Sistine Chapel ceiling is significant for ...,Michelangelo's masterpiece\nThe Sistine Chape...,Michelangelo's frescoes depict the Last Judgm...
4,What is the difference between weather and cli...,Weather refers to the short-term atmospheric c...,"The distinction is correct, but I should empha...",Weather is the short-term atmospheric conditio...,Weather refers to the temporary and local con...,Weather refers to short-term atmospheric cond...


In [ ]:
# import os, json

# from groq import Groq

# client = Groq(
#     api_key=GROQ_API_KEY,
# )

# def generate_args(question, correct_answer, standard_response, finetuned_response):
#     args = {
#         "model": "llama-3.3-70b-versatile",
#         "temperature": 0.2,
#         "messages": [
#             {
#                 "role": "system",
#                 "content": "You are an expert evaluator tasked with comparing responses from two AI systems (Model A and Model B) to the same question, and judging how close each is to the correct answer. Consider the following criteria for evaluation: - Accuracy and factual correctness compared to the ground truth - Relevance to the question asked - Completeness relative to the correct answer - Clarity and coherence - Reasoning quality - Nuance and consideration of different perspectives - Helpfulness to the user"
#             },
#             {
#                 "role": "user",
#                 "content": f"""
#                 Question: {question} 
#                 Correct Answer: {correct_answer} 
#                 Model A response: {standard_response} 
#                 Model B response: {finetuned_response} 
#                 Please evaluate both model responses on a scale of 1-10, where 1 means extremely far from the correct answer and 10 means perfectly aligned with the correct answer. First, analyze both model responses separately, comparing each to the correct answer. Identify how closely each aligns with the ground truth according to the criteria above. Then, explain specifically what each model did well or poorly compared to the correct answer. Note which model came closer to providing the correct information and why. Finally, provide a numerical score (1-10) for each model: 
#                 """
#             }
#         ],
#         "tools": [
#             {
#                 "type": "function",
#                 "function": {
#                     "name": "evaluate_models",
#                     "description": "Evaluate the two model responses and return their scores based on the evaluation criteria.",
#                     "parameters": {
#                         "type": "object",
#                         "properties": {
#                             "model_A_score": {
#                                 "type": "int",
#                                 "description": "The score of model A (Out of 10)"
#                             },
#                             "model_B_score": {
#                                 "type": "int",
#                                 "description": "The score of model B (Out of 10)"
#                             },
#                         },
#                         "required": ["model_A_score", "model_B_score"]
#                     }
#                 }
#             }
#         ],
#         "tool_choice": "auto",
#         "max_completion_tokens": 4096
#     }

#     comp = client.chat.completions.create(**args)

#     A = json.loads(comp.choices[0].message.tool_calls[0].function.arguments)['model_A_score']
#     B = json.loads(comp.choices[0].message.tool_calls[0].function.arguments)['model_B_score']
                      
#     return A, B


In [ ]:
# # for index, row in tqdm(comparison_df.iterrows()):
# model = 

# for index, row in tqdm(comparison_df.iterrows(), desc="Generating scores...", unit="question"):
#     A, B = generate_args(
#         question=row['task'],
#         correct_answer=row['answer'],
#         model_response=row['']
#     )
#     standard_model_score.append(A)
#     finetuned_model_score.append(B)

In [38]:
import os, json

from groq import Groq

# GROQ_API_KEY = 'gsk_REDACTED_ROTATE_THIS_TOKEN' # AQI

# GROQ_API_KEY = 'gsk_REDACTED_ROTATE_THIS_TOKEN' # 30401

GROQ_API_KEY = 'gsk_REDACTED_ROTATE_THIS_TOKEN' # 30402

client = Groq(
    api_key=GROQ_API_KEY,
)

def generate_args(question, correct_answer, model_response):
    args = {
        "model": "llama-3.3-70b-versatile",
        "temperature": 0.2,
        "messages": [
            {
                "role": "system",
                "content": """
You are an advanced AI model designed to critically evaluate and score model-generated answers with extreme precision. Your task is to assess responses from a Large Language model by comparing them to a ground truth answer.

Evaluation Criteria:

For each given task, evaluate the model based on the following factors:
	1.	Accuracy (0-10): How factually correct is the response compared to the ground truth?
	2.	Completeness (0-10): Does the response fully cover the key aspects of the answer?
	3.	Clarity & Coherence (0-10): Is the response well-structured, easy to understand, and logically coherent?
	4.	Conciseness & Relevance (0-10): Does the response avoid unnecessary information while staying relevant?
	5.	Depth of Reasoning (0-10): Does the response demonstrate deep understanding, including nuanced insights or self-reflection?

"""
            },
            {
                "role": "user",
                "content": f"""

	•	Compare model responses to the provided ground truth answer.
	•	Assign scores (0-10) for each category above and calculate an overall score (average of all categories).
	•	Here's the data

                Question: {question} \n
                Ground Truth Answer: {correct_answer} \n
                Model's answer: {model_response} 
            
                """
            }
        ],
        "tools": [
            {
                "type": "function",
                "function": {
                    "name": "evaluate_models",
                    "description": "Evaluate the model response and return its scores based on the evaluation criteria.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "model_score": {
                                "type": "int",
                                "description": "The score (Out of 10)"
                            },
                        },
                        "required": ["model_score"]
                    }
                }
            }
        ],
        "tool_choice": "auto",
        "max_completion_tokens": 4096
    }

    comp = client.chat.completions.create(**args)

    score = json.loads(comp.choices[0].message.tool_calls[0].function.arguments)['model_score']
                      
    return score


In [10]:
import pandas as pd

tmp = pd.read_csv('testing_model_performances.csv')
tmp

,task,reasoning,self_reflection,answer,model,finetuned_model
0,What is the longest river in the world?,The Nile River is often cited as the longest r...,I will include that some experts argue about t...,"Traditionally, the Nile River is considered th...",The Nile River is often cited as the longest ...,The longest river in the world is the Nile Ri...
1,How do vaccines work?,Vaccines work by stimulating the immune system...,I need to ensure that the mechanism of buildin...,Vaccines introduce a harmless component of a p...,\nVaccines are made from inactivated or weake...,Vaccines protect our bodies by introducing a ...
2,What is gravity?,Gravity is the force that pulls objects toward...,Correct. I could also mention it was formulate...,Gravity is the force that attracts objects tow...,Gravity is a fundamental force of nature that...,Gravity is a fundamental force of nature that...
3,What is the significance of the Sistine Chapel...,"The Sistine Chapel ceiling, painted by Michela...",Referencing both the artist and the subject ma...,The Sistine Chapel ceiling is significant for ...,Michelangelo's work on the ceiling is widely ...,Michelangelo's frescoes are renowned for thei...
4,What is the difference between weather and cli...,Weather refers to the short-term atmospheric c...,"The distinction is correct, but I should empha...",Weather is the short-term atmospheric conditio...,Weather refers to short-term atmospheric cond...,Weather refers to the short-term atmospheric ...
...,...,...,...,...,...,...
100,What is the boiling point of water in Celsius?,The boiling point of water is 100 degrees Cels...,100 degrees Celsius is correct for the boiling...,The boiling point of water is 100 degrees Cels...,\nThe boiling point of water in Celsius is 10...,180°C. The boiling point of water is 180°C.
101,What is a virus?,A virus is a non-living infectious agent that ...,Clarify it’s acellular and differs from bacteria.,A virus is an acellular infectious agent that ...,"A virus is a small, infectious particle that ...",A virus is a small infectious particle that r...
102,What is the function of chlorophyll in plants?,Chlorophyll is the pigment responsible for abs...,I need to emphasize its crucial role in captur...,"Chlorophyll absorbs sunlight, which is then us...",Chlorophyll is a green pigment that plays a c...,Chlorophyll is green pigment found in chlorop...
103,What is the difference between a concave and c...,A concave lens is thinner in the middle and di...,I should clarify that concave lenses are used ...,"A concave lens, used in glasses for nearsighte...",?\nA) The concave lens is thicker than the con...,"In a concave lens, light rays converge, while..."


In [26]:
model_scores = []

In [41]:
len(model_scores)

26

In [42]:
for index, row in tqdm(tmp[27:].iterrows(), desc="Generating scores...", unit="question"):
    task = row['task']
    
    score = generate_args(
        question=task,
        correct_answer=row['answer'],
        model_response=row['finetuned_model']
    )

    print(score)

    model_scores.append({'question': task, 'score': score})

Generating scores...: 0question [00:09, ?question/s]


KeyboardInterrupt: 